In [ ]:
from google.colab import files
import os, math, time, csv
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

# ── Class definitions (must come before torch.load) ───────────────────────────

@dataclass
class GPTConfig:
    vocab_size       : int   = None
    max_len          : int   = 256
    d_model          : int   = 256
    n_heads          : int   = 8
    n_layers         : int   = 6
    dropout          : float = 0.1
    attn_backend     : str   = "flash"
    pos_emb_type     : str   = "learned"
    batch_size       : int   = 64
    lr               : float = 3e-4
    weight_decay     : float = 0.1
    grad_clip        : float = 1.0
    max_iters        : int   = 20_000
    warmup_iters     : int   = 500
    grad_accum_steps : int   = 1
    eval_interval    : int   = 200
    eval_iters       : int   = 50
    sample_interval  : int   = 2000
    sample_length    : int   = 300
    checkpoint_dir   : str   = "checkpoints"
    checkpoint_interval : int = 2000
    train_frac       : float = 0.9

    @property
    def effective_batch_size(self):
        return self.batch_size * self.grad_accum_steps


class CharTokenizer:
    def __init__(self, text):
        self.vocab      = sorted(set(text))
        self.vocab_size = len(self.vocab)
        self.stoi       = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos       = {i: ch for i, ch in enumerate(self.vocab)}

    def encode(self, text):
        return [self.stoi[ch] for ch in text]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)


class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return x + self.embedding(positions)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe           = torch.zeros(max_len, d_model)
        position     = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div_term     = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float)
                                 * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[:d_model // 2])
        self.register_buffer("pe", pe)

    def forward(self, x):
        return self.dropout(x + self.pe[: x.size(1)])


class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_len=4096):
        super().__init__()
        inv_freq = 1.0 / (10000.0 ** (
            torch.arange(0, head_dim, 2, dtype=torch.float) / head_dim))
        self.register_buffer("inv_freq", inv_freq)
        t   = torch.arange(max_len, dtype=inv_freq.dtype)
        emb = torch.cat([torch.outer(t, inv_freq)] * 2, dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None])
        self.register_buffer("sin_cached", emb.sin()[None, None])

    @staticmethod
    def _rotate_half(x):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat([-x2, x1], dim=-1)

    def apply(self, q, k, seq_len):
        cos = self.cos_cached[:, :, :seq_len]
        sin = self.sin_cached[:, :, :seq_len]
        return (q * cos + self._rotate_half(q) * sin,
                k * cos + self._rotate_half(k) * sin)


class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, attn_backend, pos_emb_type):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads      = n_heads
        self.head_dim     = d_model // n_heads
        self.attn_backend = attn_backend
        self.dropout_p    = dropout
        self.qkv_proj     = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj     = nn.Linear(d_model, d_model,     bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.rope         = (RotaryPositionalEmbedding(self.head_dim)
                             if pos_emb_type == "rope" else None)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv_proj(x).split(C, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        if self.rope is not None:
            q, k = self.rope.apply(q, k, T)
        if self.attn_backend == "flash":
            out = F.scaled_dot_product_attention(
                q, k, v,
                dropout_p=self.dropout_p if self.training else 0.0,
                is_causal=True,
            )
        else:
            scale  = 1.0 / math.sqrt(self.head_dim)
            scores = (q @ k.transpose(-2, -1)) * scale
            mask   = torch.ones(T, T, device=x.device, dtype=torch.bool).tril()
            scores = scores.masked_fill(~mask, float("-inf"))
            out    = self.attn_dropout(F.softmax(scores, dim=-1)) @ v
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))


class FeedForward(nn.Module):
    def __init__(self, d_model, expansion=4, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, expansion * d_model, bias=False),
            nn.GELU(),
            nn.Linear(expansion * d_model, d_model, bias=False),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout, attn_backend, pos_emb_type):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadCausalSelfAttention(
            d_model, n_heads, dropout, attn_backend, pos_emb_type)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = FeedForward(d_model, dropout=dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.ff(self.ln2(x)))
        return x


class GPTDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config    = config
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        if config.pos_emb_type == "learned":
            self.pos_emb = LearnedPositionalEmbedding(config.max_len, config.d_model)
        elif config.pos_emb_type == "sinusoidal":
            self.pos_emb = SinusoidalPositionalEncoding(
                config.max_len, config.d_model, config.dropout)
        else:
            self.pos_emb = None
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.dropout,
                             config.attn_backend, config.pos_emb_type)
            for _ in range(config.n_layers)
        ])
        self.ln_f    = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("out_proj.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * config.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        x      = self.token_emb(idx)
        if self.pos_emb is not None:
            x  = self.pos_emb(x)
        for block in self.blocks:
            x  = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss   = (F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
                  if targets is not None else None)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond  = idx[:, -self.config.max_len:]
            logits, _ = self(idx_cond)
            logits    = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _  = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            idx = torch.cat([idx, torch.multinomial(
                F.softmax(logits, dim=-1), num_samples=1)], dim=1)
        return idx

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

print("Classes defined ✓")

# ── Upload and load checkpoint ────────────────────────────────────────────────
uploaded_ckpt = files.upload()
ckpt_path     = list(uploaded_ckpt.keys())[0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt   = torch.load(ckpt_path, map_location=device, weights_only=False)
config = ckpt["config"]

tokenizer            = CharTokenizer.__new__(CharTokenizer)
tokenizer.vocab      = ckpt["tokenizer_vocab"]
tokenizer.vocab_size = len(tokenizer.vocab)
tokenizer.stoi       = {ch: i for i, ch in enumerate(tokenizer.vocab)}
tokenizer.itos       = {i: ch for i, ch in enumerate(tokenizer.vocab)}

model = GPTDecoder(config).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()

print(f"✓ Model loaded from step {ckpt['step']}")
print(f"  Device     : {device}")
print(f"  Vocab size : {tokenizer.vocab_size}")
print(f"  Parameters : {model.num_parameters():,}")

# ── Generation function ───────────────────────────────────────────────────────
def generate_from_prompt(prompt, n_chars=400, temperature=0.8, top_k=40):
    unknown = [ch for ch in prompt if ch not in tokenizer.stoi]
    if unknown:
        raise ValueError(f"Characters not in vocab: {set(unknown)}")
    ctx = torch.tensor(tokenizer.encode(prompt), dtype=torch.long,
                       device=device).unsqueeze(0)
    with torch.no_grad():
        out = model.generate(ctx, n_chars, temperature, top_k)
    return tokenizer.decode(out[0].tolist())

# ── Experiments ───────────────────────────────────────────────────────────────
PROMPT = "The night is"   # ← change this

experiments = [
    {"temperature": 0.5, "top_k": 20,   "label": "Conservative (t=0.5, k=20)"},
    {"temperature": 0.8, "top_k": 40,   "label": "Balanced    (t=0.8, k=40)"},
    {"temperature": 1.2, "top_k": None, "label": "Creative    (t=1.2, k=None)"},
]

for exp in experiments:
    print(f"\n{'═'*60}")
    print(f"▶ {exp['label']}  |  prompt: '{PROMPT}'")
    print('═'*60)
    try:
        print(generate_from_prompt(PROMPT, n_chars=400,
                                   temperature=exp["temperature"],
                                   top_k=exp["top_k"]))
    except ValueError as e:
        print(f"  ⚠ {e}")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
Classes defined ✓


Saving best.pt to best.pt
✓ Model loaded from step 15800
  Device     : cuda
  Vocab size : 85
  Parameters : 4,812,544

════════════════════════════════════════════════════════════
▶ Conservative (t=0.5, k=20)  |  prompt: 'The night is'
════════════════════════════════════════════════════════════
The night is warmer;
And the purple and the flower is flown,
And the shadows are mute.

But the wind is cold and still,
And the snow is over me.

The moon is willing the wind,
And the wind is willing the rill;
And the storm is closing the rill;
And the long song is low and the wind,
And the river is long ago.

The river is willing and dim,
And the river is low;
And the river it is low;
And the river is low.



════════════════════════════════════════════════════════════
▶ Balanced    (t=0.8, k=40)  |  prompt: 'The night is'
════════════════════════════════════════════════════════════
The night is under the sun and the tree,
And the shepherd knoweth.

Some come in the gloom of the gloom,
Some 

In [ ]:
import random

# A few different prompts to get variety
prompts = [
    "The night is",
    "When silence falls",
    "I have seen the",
    "In the dark of",
    "The wind that blows",
    "The",
    "I",
    "A"
]

generated_poems = []

for i, prompt in enumerate(prompts):
    print(f"\n── Poem {i+1} ──────────────────────")
    poem = generate_from_prompt(
        prompt,
        n_chars=100000,
        temperature=0.8,
        top_k=40
    )
    generated_poems.append(poem)
    print(poem)

# Save everything to a single string for analysis
all_generated_text = "\n\n".join(generated_poems)

# Optionally save to a file
with open("generated_poems.txt", "w") as f:
    f.write(all_generated_text)

print(f"\n✓ Generated {len(generated_poems)} poems")

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
And the morn was faint and clean, and the morn was dim.

I saw, and I was never was more charming than a while,
And I loved the gaudy Maiden Stowers, and in Aire,
And now the windows and I gaped behind, and now I was a wayward side set,
And now I came to stop at Indian store the stormy night.

III

The Poet was calm, the poet's awful Hands
Around the spectral roads; and the Stars rised on the meadows
Toward the party shore; and so she smiled,
And it struck the heavy only dew, and the hearts and the lights were pale,
And her harvest smiling to the nothing more.

And her little heart was moved by the light of the sun, and the wild frost,
And the mountains looked round the shoals of the sun,
The shapes are sinking down in the sea, and the shadows of the sea,
And the shadows with those shadows that brighten and allure
Sleep in the sweet seasons, but as I wake to depart the season
Of my rough nest, mounting the se